In [ ]:
# ライブラリ

In [2]:
import os
import glob
from IPython.display import display
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
#import japanize_matplotlib
import seaborn as sns 
import sweetviz as sv
import yaml

/Users/ishizuka/opt/anaconda3/envs/mufgcup2024/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# configの読み込み
CONFIG_FILE = '/Users/ishizuka/pyworks/Competitions/signate_mufgcup2024/configs/config.yaml'
with open(CONFIG_FILE, encoding="utf-8") as file:
    yml = yaml.safe_load(file)

In [8]:
DIR_INPUT = yml["SETTING"]["DIR_INPUT"]
FILE_NAME_STATION = yml["SETTING"]["FILE_NAME_STATION"]
FILE_NAME_STATUS = yml["SETTING"]["FILE_NAME_STATUS"]
FILE_NAME_TRIP = yml["SETTING"]["FILE_NAME_TRIP"]
FILE_NAME_WEATHER = yml["SETTING"]["FILE_NAME_WEATHER"]
DIR_FIGURE = yml["SETTING"]["DIR_FIGURE"]

In [9]:
pd.set_option("display.max_columns",100)
pd.set_option("display.max_rows", 500)

# データ読み込み

In [10]:
df_station = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_STATION))
df_status = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_STATUS))
df_trip = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_TRIP))
df_weather = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_WEATHER))

In [11]:
print(df_station.shape)
print(df_status.shape)
print(df_trip.shape)
print(df_weather.shape)

(70, 6)
(1226400, 8)
(669959, 8)
(730, 22)


In [13]:
df_station.head()

,station_id,lat,long,dock_count,city,installation_date
0,0,37.32973,-121.90178,27,city1,8/6/2013
1,1,37.33070,-121.88898,15,city1,8/5/2013
2,2,37.33399,-121.89490,11,city1,8/6/2013
3,3,37.33141,-121.89320,19,city1,8/5/2013
4,4,37.33672,-121.89407,15,city1,8/7/2013


In [14]:
df_status.head()

,id,year,month,day,hour,station_id,bikes_available,predict
0,0,2013,9,1,0,0,11.0,0
1,1,2013,9,1,1,0,11.0,0
2,2,2013,9,1,2,0,11.0,0
3,3,2013,9,1,3,0,11.0,0
4,4,2013,9,1,4,0,11.0,0


In [15]:
df_trip.head()

,trip_id,duration,start_date,start_station_id,end_date,end_station_id,bike_id,subscription_type
0,0,63,8/29/2013 14:13,54,8/29/2013 14:14,54,0,Subscriber
1,1,70,8/29/2013 14:42,8,8/29/2013 14:43,8,1,Subscriber
2,2,71,8/29/2013 10:16,20,8/29/2013 10:17,20,2,Subscriber
3,3,77,8/29/2013 11:29,8,8/29/2013 11:30,8,3,Subscriber
4,4,83,8/29/2013 12:02,54,8/29/2013 12:04,55,4,Subscriber


In [16]:
df_weather.head()

,date,max_temperature,mean_temperature,min_temperature,max_dew_point,mean_dew_point,min_dew_point,max_humidity,mean_humidity,min_humidity,max_sea_level_pressure,mean_sea_level_pressure,min_sea_level_pressure,max_visibility,mean_visibility,min_visibility,max_wind_Speed,mean_wind_speed,precipitation,cloud_cover,events,wind_dir_degrees
0,2013-09-01,81,70,61,62,58,54,80,67,47,29.94,29.95,29.85,10,10,10,14,4,0.00,1,NaN,354
1,2013-09-02,80,71,66,64,61,58,80,70,58,29.95,29.95,29.86,10,10,10,14,4,0.00,5,NaN,337
2,2013-09-03,81,69,58,60,56,52,82,65,44,29.99,29.99,29.93,10,10,10,19,2,1.71,6,Rain,341
3,2013-09-04,82,68,56,61,55,49,81,64,43,30.04,30.02,29.94,10,10,10,15,0,0.00,0,NaN,324
4,2013-09-05,81,68,56,59,54,50,81,63,41,30.02,30.02,29.95,10,10,10,16,1,0.00,0,NaN,335


# 型、項目数、欠損値率

In [25]:
def calc_describe(df):
    dtypes = []
    val_counts_station = []
    isnull_station = []
    isnull_station_ratio = 100 * df.isnull().sum() / len(df)
    for col in df.columns:
        dtypes.append(str(df[col].dtype))
        val_counts_station.append(len(df[col].value_counts()))
        isnull_station.append(isnull_station_ratio[col])
    inds = ["型", "val_counts", "NaN率"]
    df_eda = pd.DataFrame([dtypes, val_counts_station, isnull_station], columns=df.columns, index=inds).T
    # df_eda.query("val_counts > 1")
    return df_eda

In [26]:
df_eda_station = calc_describe(df_station)
df_eda_station

,型,val_counts,NaN率
station_id,int64,70,0.0
lat,float64,69,0.0
long,float64,70,0.0
dock_count,int64,6,0.0
city,object,5,0.0
installation_date,object,17,0.0


In [27]:
df_eda_status = calc_describe(df_status)
df_eda_status

,型,val_counts,NaN率
id,int64,1226400,0.0
year,int64,3,0.0
month,int64,12,0.0
day,int64,31,0.0
hour,int64,24,0.0
station_id,int64,70,0.0
bikes_available,float64,28,17.818167
predict,int64,2,0.0


In [28]:
df_eda_trip = calc_describe(df_trip)
df_eda_trip

,型,val_counts,NaN率
trip_id,int64,669959,0.0
duration,int64,16129,0.0
start_date,object,361559,0.0
start_station_id,int64,70,0.0
end_date,object,357757,0.0
end_station_id,int64,70,0.0
bike_id,int64,700,0.0
subscription_type,object,2,0.0


In [29]:
df_eda_weather = calc_describe(df_weather)
df_eda_weather

,型,val_counts,NaN率
date,object,730,0.0
max_temperature,int64,48,0.0
mean_temperature,int64,41,0.0
min_temperature,int64,37,0.0
max_dew_point,int64,39,0.0
mean_dew_point,int64,39,0.0
min_dew_point,int64,47,0.0
max_humidity,int64,43,0.0
mean_humidity,int64,55,0.0
min_humidity,int64,65,0.0


# 基本統計量

In [30]:
df_station.describe()

,station_id,lat,long,dock_count
count,70.000000,70.000000,70.000000,70.000000
mean,34.500000,37.590244,-122.218416,17.657143
std,20.351085,0.203473,0.209446,4.010442
min,0.000000,37.329730,-122.418950,11.000000
25%,17.250000,37.389485,-122.400600,15.000000
50%,34.500000,37.631165,-122.312120,15.000000
75%,51.750000,37.788125,-122.078008,19.000000
max,69.000000,37.804770,-121.877350,27.000000


In [31]:
df_status.describe()

,id,year,month,day,hour,station_id,bikes_available,predict
count,1.226400e+06,1.226400e+06,1.226400e+06,1.226400e+06,1.226400e+06,1.226400e+06,1.007878e+06,1.226400e+06
mean,6.131995e+05,2.014166e+03,6.526027e+00,1.572055e+01,1.150000e+01,3.450000e+01,8.428755e+00,1.575342e-01
std,3.540313e+05,6.874054e-01,3.447853e+00,8.796251e+00,6.922189e+00,2.020521e+01,3.953576e+00,3.643039e-01
min,0.000000e+00,2.013000e+03,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.065998e+05,2.014000e+03,4.000000e+00,8.000000e+00,5.750000e+00,1.700000e+01,6.000000e+00,0.000000e+00
50%,6.131995e+05,2.014000e+03,7.000000e+00,1.600000e+01,1.150000e+01,3.450000e+01,8.000000e+00,0.000000e+00
75%,9.197992e+05,2.015000e+03,1.000000e+01,2.300000e+01,1.725000e+01,5.200000e+01,1.100000e+01,0.000000e+00
max,1.226399e+06,2.015000e+03,1.200000e+01,3.100000e+01,2.300000e+01,6.900000e+01,2.700000e+01,1.000000e+00


In [32]:
df_trip.describe()

,trip_id,duration,start_station_id,end_station_id,bike_id
count,669959.000000,6.699590e+05,669959.000000,669959.000000,669959.00000
mean,334979.000000,1.107950e+03,47.144845,47.177594,281.98319
std,193400.648836,2.225544e+04,14.669843,14.733621,186.50497
min,0.000000,6.000000e+01,0.000000,0.000000,0.00000
25%,167489.500000,3.440000e+02,40.000000,39.000000,127.00000
50%,334979.000000,5.170000e+02,50.000000,50.000000,254.00000
75%,502468.500000,7.550000e+02,58.000000,58.000000,421.00000
max,669958.000000,1.727040e+07,69.000000,69.000000,699.00000


In [33]:
df_weather.describe()

,max_temperature,mean_temperature,min_temperature,max_dew_point,mean_dew_point,min_dew_point,max_humidity,mean_humidity,min_humidity,max_sea_level_pressure,mean_sea_level_pressure,min_sea_level_pressure,max_visibility,mean_visibility,min_visibility,max_wind_Speed,mean_wind_speed,precipitation,cloud_cover,wind_dir_degrees
count,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000
mean,71.424658,61.357534,51.582192,53.204110,48.554795,43.812329,83.745205,65.858904,46.334247,30.081671,30.031274,29.967479,9.994521,9.654795,8.439726,16.742466,4.763014,0.030329,1.836986,270.158904
std,8.185802,7.683171,7.913071,6.993294,7.960319,9.411146,8.646749,9.439466,13.068903,0.132638,0.130428,0.131095,0.073871,0.892745,2.667142,7.314190,2.510882,0.187340,2.062002,118.615872
min,46.000000,38.000000,28.000000,23.000000,15.000000,8.000000,40.000000,28.000000,11.000000,29.500000,29.440000,29.360000,9.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,66.000000,56.000000,46.000000,49.000000,44.000000,37.000000,80.000000,61.000000,38.000000,29.982500,29.940000,29.880000,10.000000,10.000000,8.000000,14.000000,2.000000,0.000000,0.000000,262.500000
50%,71.000000,62.000000,52.000000,54.000000,49.000000,45.000000,84.000000,67.000000,47.000000,30.070000,30.020000,29.960000,10.000000,10.000000,10.000000,16.000000,5.000000,0.000000,1.000000,331.000000
75%,77.000000,68.000000,58.000000,59.000000,55.000000,51.000000,89.000000,72.000000,55.000000,30.170000,30.120000,30.060000,10.000000,10.000000,10.000000,20.000000,7.000000,0.000000,3.000000,345.000000
max,99.000000,82.000000,66.000000,66.000000,62.000000,60.000000,100.000000,92.000000,84.000000,30.480000,30.410000,30.350000,10.000000,10.000000,10.000000,114.000000,14.000000,3.360000,8.000000,360.000000


# ID

In [ ]:
# インデックスと同様

# Age

表記揺れあり

In [23]:
df_all["Age"].value_counts()

Age
30代     224
33歳     194
36歳     188
40代     184
37歳     182
32歳     176
35歳     167
34歳     163
31歳     154
40歳     151
38歳     150
30歳     146
42歳     146
39歳     142
41歳     132
43歳     124
50歳     118
45歳     117
50代     117
52歳     114
51歳     113
28歳     111
49歳     106
47歳     104
44歳     104
29歳     101
46歳     100
53歳      95
55歳      92
26歳      88
20代      87
54歳      87
48歳      86
27歳      84
56歳      80
25歳      77
24歳      69
57歳      65
22歳      58
23歳      56
58歳      47
20歳      37
21歳      37
59歳      36
34才      29
３３歳      27
36才      26
３０代      24
31才      24
29才      22
60歳      21
３７歳      21
39才      20
３５歳      20
４３歳      19
４０代      19
28才      19
３６歳      19
３９歳      19
３８歳      19
32才      18
３２歳      18
51才      18
３４歳      18
19歳      17
48才      17
三十三歳     17
46際      17
27才      16
33才      16
３１歳      16
38才      15
41才      15
３０歳      15
４５歳      15
４２歳      15
35才      15
46才      15
50才      15
37才      14
42才      14
49才      14
４１歳      14


# DurationOfPitch

In [29]:
df_all["DurationOfPitch"].unique()

array(['900秒', '14分', '10分', '1080秒', '1020秒', '16分', '840秒', '22分',
       '20分', '11分', '32分', '17分', '24分', '1380秒', '35分', '420秒', '5分',
       '480秒', '8分', '15分', '960秒', nan, '540秒', '26分', '13分', '12分',
       '21分', '25分', '720秒', '9分', '7分', '780秒', '6分', '18分', '31分',
       '600秒', '23分', '28分', '2160秒', '660秒', '1500秒', '1920秒', '27分',
       '33分', '360秒', '1320秒', '1740秒', '1680秒', '29分', '1620秒', '1440秒',
       '1800秒', '30分', '34分', '4分', '19分', '1860秒', '1260秒', '2100秒',
       '300秒', '2040秒', '1200秒', '1560秒', '1980秒', '36分', '1140秒', '240秒'],
      dtype=object)

In [28]:
df_all["DurationOfPitch"].value_counts()

DurationOfPitch
8分       495
9分       488
15分      432
16分      408
14分      398
10分      366
7分       322
13分      299
11分      233
17分      230
12分      206
480秒     176
540秒     166
900秒     137
6分       133
420秒     124
840秒     120
960秒     119
600秒     101
780秒      98
18分       97
720秒      87
1020秒     82
23分       76
32分       75
660秒      73
24分       67
31分       61
22分       60
25分       59
34分       55
26分       51
33分       50
20分       50
30分       49
21分       48
28分       42
27分       42
19分       41
35分       41
1080秒     36
360秒      34
29分       28
1380秒     28
5分        25
1860秒     25
1320秒     24
1740秒     20
1500秒     20
2040秒     20
1980秒     19
1440秒     19
300秒      18
1920秒     18
1260秒     17
2100秒     17
1800秒     16
1200秒     16
1620秒     15
1680秒     14
1560秒     13
36分       10
1140秒      9
2160秒      5
4分         2
240秒       1
Name: count, dtype: int64

# Gender

In [31]:
df_all["Gender"].value_counts()

Gender
Male       2525
Female     1441
male        940
female      504
MALE        363
Ｍａｌｅ        260
Fe Male     213
FEMALE      197
Ｆｅｍａｌｅ      181
ｍａｌｅ         92
ｆｅｍａｌｅ       65
fe male      57
FE MALE      43
ＭＡＬＥ         38
Ｆｅ　Ｍａｌｅ      26
ＦＥＭＡＬＥ       21
ｆｅ　ｍａｌｅ      11
ＦＥ　ＭＡＬＥ       1
Name: count, dtype: int64

# NumberOfFollowups

In [33]:
df_all["NumberOfFollowups"].value_counts()

NumberOfFollowups
4.0      2746
3.0      2536
5.0      1113
2.0       212
1.0       176
6.0        69
400.0      31
300.0      25
500.0       9
100.0       2
600.0       1
200.0       1
Name: count, dtype: int64

In [38]:
df_all.groupby("NumberOfFollowups").agg("ProdTaken").mean()

NumberOfFollowups
1.0      0.129870
2.0      0.055556
3.0      0.167318
4.0      0.134670
5.0      0.113680
6.0      0.368421
100.0    0.000000
200.0         NaN
300.0    0.117647
400.0    0.066667
500.0    0.200000
600.0    0.000000
Name: ProdTaken, dtype: float64

# ProductPitched

In [40]:
df_all["ProductPitched"].unique()

array(['Basic', 'Standard', 'Super Deluxe', 'basic', 'SUPER DELUXE',
       'super deluxe', 'BASIC', 'Deluxe', 'deluxe', 'STANᗞARD',
       'STANDARD', 'Вasic', 'DELUXE', 'ꓢuper De|uxe', 'Ѕuper Deluxe',
       'BAՏIC', 'Basıc', 'King', 'Super De|uxe', 'king', 'standard',
       'KING', 'BΑSIC', 'B𝖺sic', 'De|uxe', 'ᎠELUXE', 'Basiϲ', 'de|u×e',
       'Delu×e', 'Standar𝔡', 'Basi𝘤', 'Βasic', 'Տuper Deluxe', 'Staոdard',
       'BAꓢIC', 'ᗞeluxe', 'Տtandard', 'Βası𝘤', 'Kıng', 'Baｓic', 'basıc',
       'super de|uxe', 'Stand𝖺rd', 'S𝘵andard', '𐊡asic', 'St𝖺ndard',
       'Super ᗞeluxe', 'de|uxe', 'ｓuper deluxe', 'STANDARᎠ', 'Basiс',
       'DΕLUXΕ', 'ꓢuper Deluxe', 'BASΙC', 'ꓢtandard', 'В𝖺sic', 'Standa𝘳d',
       'basiϲ', 'staոdard', 'Super Ꭰeluxe', 'DELUXΕ', 'Ѕtandard', '𐊡asi𝘤',
       'Ꭰeluxe', 'Kıոg', '𝙳eluxe', 'Kiոg', 'Βasıc', 'BASIС',
       'SUPER DΕLUXE', 'B𝖺si𝘤', 'ΒASIС', 'Super 𝙳eluxe', 'Տtanda𝘳d',
       'Basıϲ', 'ЅTANDARD', 'SUPER ᎠELUXE', 'SUPER ᗞELUXE', 'basiс',
       'Stan𝔡ard', 'S

In [39]:
df_all["ProductPitched"].value_counts()

ProductPitched
Basic           1810
Deluxe          1660
Standard        1239
Super Deluxe     471
basic            218
King             217
BASIC            212
DELUXE           191
deluxe           179
STANDARD         164
standard         153
super deluxe      75
SUPER DELUXE      58
KING              35
Basıc             31
king              29
De|uxe            26
Βasic             10
Delu×e             9
Super De|uxe       9
Ѕtandard           7
ᗞeluxe             7
Basi𝘤              7
Basiϲ              6
Standa𝘳d           6
Baｓic              6
Basiс              6
Stand𝖺rd           6
Staոdard           5
Вasic              5
Super Ꭰeluxe       5
𐊡asic              5
St𝖺ndard           5
Super ᗞeluxe       5
ꓢtandard           4
𝙳eluxe             4
Ꭰeluxe             4
Տtandard           3
DELUXΕ             3
basiϲ              3
B𝖺sic              3
S𝘵andard           3
Stan𝔡ard           3
Super 𝙳eluxe       3
ꓢuper Deluxe       3
ᗞELUXE             2
STΑNDARD           

# NumberOfTrips

In [42]:
df_all["NumberOfTrips"].value_counts()

NumberOfTrips
2         1956
3         1461
5          836
1          669
7          445
年に2回       294
4          277
年に3回       245
6          234
年に5回       141
年に1回       132
年に7回        68
年に4回        50
年に6回        47
半年に1回       27
8           20
年に8回         6
四半期に1回       6
Name: count, dtype: int64

# Designation

In [45]:
df_all["Designation"].unique()

array(['Executive', 'Senior Manager', 'AVP', 'Manager', 'Senior Manage𝙧',
       'Execuｔive', 'Μanager', 'VP', 'Sеnior Manager', 'ΑVP', 'АVP',
       'E×ecutive', 'Mαnage𝙧', 'Executiѵе', 'Ѕenior Manager', 'Managеr',
       'Еxecutivе', 'Senior Μanαger', 'Еxecuｔive', 'Exеcutivе',
       'Exеcutive', 'Senior Managе𝙧', 'Manage𝙧', 'Senio𝙧 Manager',
       'Manαger', 'Μanage𝙧', 'Manαgеr', 'Senior Managеr', 'Executivе',
       'Executiѵe', 'Е×еcutive', 'Еxecutive', 'VＰ', 'Տenior Μanager',
       'Exеcutiѵе', 'Senior Manαger', 'Mαnager', 'Senior Mαnαger',
       'E×еcutiѵe', 'Ѕenior Manαger', 'Exеcｕtive', 'Execｕtive', 'Mαnαger',
       'Μanagеr', 'E×ecｕtive', 'Sеnior Managеr', 'Տenior Manager', 'AVＰ',
       'Exеcｕtivе', 'Mαnagеr', 'Еxеcutivе', 'Senior Mαnager', 'Е×ecutive',
       'Senio𝙧 Manage𝙧', 'ΑVＰ', 'Μαnager', 'Senio𝙧 Manαger',
       'Ѕenior Μanage𝙧', 'Exеcuｔive', 'Μαnagеr', 'Execｕｔive', 'Managе𝙧',
       'Senio𝙧 Managеr', 'Senior Μanager', 'Sеnior Managе𝙧', 'Execｕtivе',
       'Senio

In [44]:
df_all["Designation"].value_counts()

Designation
Executive         2219
Manager           2027
Senior Manager    1565
AVP                604
VP                 259
Exеcutive           19
Exеcutivе           19
Μanager             18
Managеr             18
Executivе           16
Manαger             16
АVP                 14
Mαnager             13
Executiѵe           13
Senior Managеr      12
ΑVP                 11
Еxecutive            8
Manage𝙧              8
Sеnior Manager       7
Senio𝙧 Manager       6
Senior Manαger       6
Execuｔive            6
Senior Manage𝙧       5
VＰ                   5
Տenior Manager       4
AVＰ                  4
Senior Mαnαger       4
Mαnαger              4
E×ecutive            4
Execｕtive            4
Ѕenior Manager       4
Exеcutiѵе            3
Exеcｕtive            2
Exеcｕtivе            2
Mαnagеr              2
Senior Mαnager       2
Μαnager              2
Senior Μanager       2
Sеnior Managеr       2
Senior Managе𝙧       2
Еxecutivе            2
Е×еcutive            2
Executiѵе            2

# MonthlyIncome

In [47]:
df_all["MonthlyIncome"].unique()

array(['253905.0', '404475.0', '278145.0', ..., '319275.0', '261840.0',
       '272430.0'], dtype=object)

In [48]:
df_all["MonthlyIncome"].value_counts()

MonthlyIncome
月収30.0万円    388
月収40.0万円    253
月収50.0万円     81
月収26.0万円     56
月収32.0万円     56
           ... 
472590.0      1
373260.0      1
245100.0      1
343290.0      1
272430.0      1
Name: count, Length: 4587, dtype: int64

# customer_info

In [50]:
df_all["customer_info"]

0           未婚 車未所持 子供なし
1          離婚済み,車あり,子供無し
2       結婚済み、自動車未所有,子供なし
3          離婚済み、車所持、子供無し
4              独身／車所持／無子
              ...       
3484       結婚済み/車なし／子供無し
3485    結婚済み、自家用車あり、子供1人
3486        独身、車未所持、子供なし
3487     結婚済み、車未所持、こども1人
3488         未婚　車なし　子供3人
Name: customer_info, Length: 6978, dtype: object